# Memória e Estado

Uma chamada ao modelo é isolada. Tudo o que ele parece lembrar está escrito no prompt daquela chamada ou ficou nos pesos durante o treino, e memória é o nome do conjunto de decisões sobre o que entra ali.

O assistente deste notebook acompanha uma mesa de RPG. Ele separa três coisas: o que veio do treino, o que vale para a sessão atual e o que sobrevive ao fim dela, guardado em um arquivo.

In [ ]:
# No Google Colab, descomente e rode uma vez (Ambiente de execução > GPU).
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

import json
from pathlib import Path

import pandas as pd
import torch

from agentkit import LLM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=100)
print(llm.model)

## Memória paramétrica

Parte do que o modelo responde não vem do prompt. O treino ajustou bilhões de parâmetros sobre um corpus enorme, e o que ficou ali é consultado de graça em toda chamada. As duas perguntas abaixo têm a mesma forma e destinos diferentes.

In [ ]:
print(llm.invoke([{"role": "user", "content": "Quantas faces tem um dado d20? Responda apenas o número."}]))
print(llm.invoke([{"role": "user", "content": "Quem é o vilão da campanha que a Bia joga às terças?"}]))

A primeira resposta veio dos pesos e está certa. A segunda trata de uma campanha que nunca esteve no corpus de treino, e o modelo reconhece a lacuna e pede contexto. Essa memória é congelada na data do treino, não é editável em tempo de uso e não registra de onde cada afirmação veio.

## Memória de curto prazo

É o que vale para a execução atual e morre com ela. Aqui estão o histórico da conversa, as políticas que decidem o que cabe no contexto e o estado que o programa carrega enquanto executa.

### Histórico

O modelo não recebe um identificador de sessão nem consulta o que foi dito antes. As duas chamadas abaixo são independentes, e a segunda não recebe a primeira.

In [ ]:
SYSTEM = {"role": "system", "content": "Você ajuda a mestrar uma mesa de RPG. Responda em uma frase curta."}

print(llm.invoke([SYSTEM, {"role": "user", "content": "Minha personagem se chama Yara."}]))
print(llm.invoke([SYSTEM, {"role": "user", "content": "Como se chama minha personagem?"}]))

Nada foi esquecido, porque nada foi guardado. O primeiro tipo de memória de um chatbot é uma lista mantida pelo programa: cada rodada acrescenta a entrada do usuário e a resposta, e a chamada seguinte recebe a lista inteira.

In [ ]:
def chat(history: list[dict], message: str) -> str:
    """Executa um turno e modifica o histórico no lugar."""
    history.append({"role": "user", "content": message})
    answer = llm.invoke(history)
    history.append({"role": "assistant", "content": answer})
    return answer


TURNS = [
    "Sou a Bia e jogo com a Yara, uma arqueira elfa.",
    "Na sessão passada o grupo entrou na caverna do norte.",
    "A Yara está com 12 pontos de vida.",
    "Ela guardou o Amuleto de Névoa na mochila.",
    "A próxima sessão é dia 14 de março.",
]

history = [SYSTEM.copy()]
for turn in TURNS:
    chat(history, turn)
pd.DataFrame(history)

In [ ]:
QUESTION = {"role": "user", "content": "Como se chama minha personagem?"}
print(llm.invoke(history + [QUESTION]))

Com a lista inteira no prompt, o nome da personagem volta. Os papéis fazem parte do significado do histórico: `system` define o comportamento, `user` traz as entradas e `assistant` registra as respostas anteriores.

### Janela deslizante

O modelo recebe todas as mensagens de novo a cada turno, então o custo cresce com o histórico. A política mais simples é cortar pelo fim, mantendo a mensagem de sistema e as últimas mensagens.

In [ ]:
def recent_messages(history: list[dict], max_messages: int = 4) -> list[dict]:
    """Preserva a mensagem de sistema e devolve as últimas mensagens."""
    return [history[0], *history[1:][-max_messages:]]


window = recent_messages(history)
print(f"histórico: {len(history)} mensagens | janela: {len(window)} mensagens")
pd.DataFrame(window)

In [ ]:
print(llm.invoke(window + [QUESTION]))

A janela ficou com os dois últimos turnos, e o nome da personagem, dito no primeiro, saiu do contexto. O corte é cego: ele olha a posição e descarta com a mesma facilidade uma saudação e o dado que a próxima pergunta precisa.

### Sumarização progressiva

Em vez de descartar o começo, podemos criar uma representação compacta dele. A conversa vai entre marcas e a instrução vem depois dela, porque sem isso o modelo continua o diálogo em vez de resumi-lo. O resumo entra dentro da mensagem de sistema que já existe.

In [ ]:
def summarize(messages: list[dict]) -> str:
    """Resume um trecho de conversa preservando nomes, valores e datas."""
    conversation = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
    return llm.invoke([{"role": "user", "content": (
        f"<conversa>\n{conversation}\n</conversa>\n\n"
        "Resuma a conversa acima em até cinco linhas, preservando nomes, valores e datas."
    )}], max_tokens=140)


summary = summarize(history[1:-4])
print(summary)

In [ ]:
summarized = [
    {"role": "system", "content": f"{SYSTEM['content']}\n\nResumo da conversa anterior:\n{summary}"},
    *history[-4:],
]
print(f"completo: {len(history)} mensagens | resumido: {len(summarized)} mensagens")

In [ ]:
CHECKS = [
    "Como se chama a personagem?",
    "Qual é a classe dela?",
    "Onde o grupo entrou na sessão passada?",
    "Quando é a próxima sessão?",
]

pd.DataFrame([{
    "pergunta": check,
    "histórico completo": llm.invoke(history + [{"role": "user", "content": check}], max_tokens=60),
    "contexto resumido": llm.invoke(summarized + [{"role": "user", "content": check}], max_tokens=60),
} for check in CHECKS])

O histórico completo responde às quatro perguntas. O resumido acerta o lugar e a data, que estão nas mensagens preservadas, e perde o nome e a classe, ditos no primeiro turno.

A instrução do resumo mandava preservar nomes e a classe ficou de fora mesmo assim. Na pergunta do lugar, o contexto curto ainda acrescentou um combate que ninguém mencionou: comprimir decide o que perder e abre espaço para o modelo preencher o resto.

### Exercício 1

Monte um assistente com histórico e converse quatro turnos, colocando um número importante no primeiro. Depois aplique a janela deslizante e responda a partir de qual turno esse número deixaria de estar no contexto.

In [ ]:
# Seu código aqui

### Estado do agente

O histórico é uma parte do estado, e não o estado inteiro. O assistente também carrega o que a sessão já estabeleceu: onde o grupo está, quanta vida a personagem tem e o que ela leva na mochila.

In [ ]:
state = {
    "personagem": "Yara",
    "local": "caverna do norte",
    "vida": 12,
    "inventario": [],
    "messages": [],
}
state

In [ ]:
def system_prompt(state: dict) -> dict:
    """Escreve a mensagem de sistema a partir do estado atual."""
    return {"role": "system", "content": (
        f"{SYSTEM['content']}\n\n"
        f"Personagem: {state['personagem']}\n"
        f"Local: {state['local']}\n"
        f"Vida: {state['vida']}\n"
        f"Inventário: {', '.join(state['inventario']) or 'vazio'}"
    )}


print(system_prompt(state)["content"])

O estado é dado, e o prompt é texto. A função acima faz a passagem de um para o outro, e é ela que decide o que da execução o modelo enxerga. A função abaixo faz o turno: recebe o estado, pergunta ao modelo e devolve o estado com a conversa acrescentada, na mesma forma do `chat` da primeira parte.

In [ ]:
def turn(state: dict, message: str) -> dict:
    """Faz um turno usando o estado como contexto e devolve o estado."""
    answer = llm.invoke([system_prompt(state), *state["messages"],
                         {"role": "user", "content": message}])
    state["messages"] += [{"role": "user", "content": message},
                          {"role": "assistant", "content": answer}]
    return state

In [ ]:
QUESTION = "Onde eu estou, como está minha vida e o que carrego?"

state = turn(state, QUESTION)
print(state["messages"][-1]["content"])

In [ ]:
state["vida"] = 8
state["local"] = "ponte quebrada"
state["inventario"].append("Amuleto de Névoa")

state = turn(state, QUESTION)
print(state["messages"][-1]["content"])
print(state["local"], "|", state["vida"], "de vida |", len(state["messages"]), "mensagens")

A mesma pergunta foi feita duas vezes e a resposta mudou, porque o prompt é montado a partir do estado e o estado mudou entre elas. As três linhas do meio são a mesa acontecendo, e o `turn` não precisa saber delas: ele lê o estado que encontrar.

Nada disso sobrevive ao fim do processo, e é esse o problema da parte seguinte.

### Exercício 2

Monte o estado de outro domínio, como um pedido de pizza com sabor, tamanho e endereço. Escreva a mensagem de sistema a partir dele, pergunte ao modelo o que já foi pedido, mude um campo e pergunte de novo. Responda o que mudou na resposta.

In [ ]:
# Seu código aqui

## Memória de longo prazo

Tudo acima morre com o processo. Guardar algo para a próxima execução exige uma estrutura fora do modelo e fora da memória do programa, e o mínimo que serve é um arquivo que o aluno pode abrir em qualquer editor.

In [ ]:
MEMORY_PATH = Path("workspace/memory.json")
MEMORY_PATH.parent.mkdir(parents=True, exist_ok=True)


def load_memory() -> list[dict]:
    """Lê a memória do disco, ou devolve uma lista vazia na primeira execução."""
    if MEMORY_PATH.exists():
        return json.loads(MEMORY_PATH.read_text(encoding="utf-8"))
    return []


def save_memory(memory: list[dict]) -> None:
    """Grava a memória inteira no disco, em JSON legível."""
    MEMORY_PATH.write_text(json.dumps(memory, ensure_ascii=False, indent=2), encoding="utf-8")

Cada registro tem três campos: o tipo, uma chave que o identifica dentro do tipo e o conteúdo. Gravar com o mesmo tipo e a mesma chave substitui o conteúdo, e é essa regra que decide sozinha o comportamento das três memórias adiante.

In [ ]:
def remember(memory: list[dict], kind: str, key: str, content: str) -> None:
    """Grava um registro, substituindo o que tiver o mesmo tipo e a mesma chave."""
    for item in memory:
        if item["type"] == kind and item["key"] == key:
            item["content"] = content
            return
    memory.append({"type": kind, "key": key, "content": content})


def recall(memory: list[dict], kind: str) -> list[dict]:
    """Devolve os registros de um tipo."""
    return [item for item in memory if item["type"] == kind]

### Memória semântica

A memória semântica guarda o que é verdade sobre o cliente, sem registro de quando foi aprendido. A chave é o nome do campo, então o mesmo fato tem uma versão só, sempre a última.

In [ ]:
memory = load_memory()
remember(memory, "semantic", "personagem", "Yara, arqueira elfa")
remember(memory, "semantic", "cidade_natal", "Vale de Anor")
remember(memory, "semantic", "patrono", "a Ordem da Aurora")
remember(memory, "semantic", "patrono", "a Guilda dos Cartógrafos")
save_memory(memory)
pd.DataFrame(recall(memory, "semantic"))

In [ ]:
facts = "\n".join(f"- {item['key']}: {item['content']}" for item in recall(memory, "semantic"))
print(llm.invoke([
    {"role": "system", "content": f"{SYSTEM['content']}\n\nFatos conhecidos:\n{facts}"},
    {"role": "user", "content": "Quem é minha personagem e a serviço de quem ela está?"},
]))

O `patrono` foi gravado duas vezes e a tabela tem uma linha só, com o valor mais recente. A atualização é propriedade da estrutura: como a chave é o nome do campo, o fato tem uma versão só, e foi a Guilda dos Cartógrafos que chegou à resposta.

### Memória episódica

A memória episódica guarda o que aconteceu, e responde a outra pergunta: o que já foi feito. A chave é a data, então dois episódios diferentes nunca colidem e a lista cresce.

In [ ]:
remember(memory, "episodic", "2026-02-28", "O grupo entrou na caverna do norte e enfrentou o guardião de pedra.")
remember(memory, "episodic", "2026-03-14", "Yara achou o Amuleto de Névoa e o mestre cobrou um teste de furtividade.")
save_memory(memory)
pd.DataFrame(recall(memory, "episodic"))

In [ ]:
episodes = "\n".join(f"- {item['key']}: {item['content']}" for item in recall(memory, "episodic"))
print(llm.invoke([
    {"role": "system", "content": f"{SYSTEM['content']}\n\nRegistros anteriores:\n{episodes}"},
    {"role": "user", "content": "O que aconteceu na sessão de fevereiro?"},
]))

Os dois episódios convivem, porque as chaves são datas diferentes e nada colidiu. A mesma função de escrita produziu o comportamento oposto ao da parte anterior, e o que mudou foi só a escolha da chave. A resposta trouxe o registro de fevereiro, que é o que a pergunta pedia.

### Memória procedural

A memória procedural guarda como fazer. Ela entra no prompt como instrução, e não como dado a consultar, e é essa a diferença de tratamento em relação às outras duas.

In [ ]:
remember(memory, "procedural", "testes", "Teste difícil se resolve com d20, e o grupo ajudando dá vantagem.")
remember(memory, "procedural", "fichas", "As fichas são atualizadas ao fim de cada sessão, nunca no meio da cena.")
save_memory(memory)
pd.DataFrame(recall(memory, "procedural"))

In [ ]:
rules = "\n".join(f"- {item['content']}" for item in recall(memory, "procedural"))
print(llm.invoke([
    {"role": "system", "content": f"{SYSTEM['content']}\n\nProcedimentos a seguir:\n{rules}"},
    {"role": "user", "content": "Como eu resolvo um teste difícil?"},
]))

A resposta seguiu a regra da mesa, e nenhuma consulta foi feita para isso. É a diferença de tratamento entre os três tipos: fato e episódio entram como dado, para o modelo consultar, e procedimento entra como instrução, para o modelo obedecer.

### Exercício 3

Carregue a memória do arquivo em uma execução nova, responda a uma pergunta usando os fatos guardados e grave um fato novo. Mostre que ele continua no arquivo depois, e responda o que aconteceria se você gravasse duas vezes na mesma chave.

In [ ]:
# Seu código aqui